Embedding Models Used:
- HuggingFace all-MiniLM-L6-v2
- Ollama nomic-embed-text

Vector Stores Used:
- FAISS
- ChromaDB


In [3]:
# PART 0 — Environment Setup

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import OllamaEmbeddings

from langchain_community.vectorstores import FAISS, Chroma

import os

In [4]:
# PART 1 — TASK 1
# Step 1: Load documents

data_path = "data"
documents = []

for file in os.listdir(data_path):
    loader = TextLoader(os.path.join(data_path, file))
    documents.extend(loader.load())

print("Loaded documents:", len(documents))


Loaded documents: 3


In [5]:
# Step 2: Split documents into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))


Total chunks: 1


In [7]:
# Step 3: Generate embeddings (HuggingFace)
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

emb = hf_embeddings.embed_query(chunks[0].page_content)

print("Embedding Vector Length:", len(emb))
print("Sample Values:", emb[:10])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 262.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Vector Length: 384
Sample Values: [-0.09544370323419571, 0.0033895375672727823, -0.00884275883436203, 0.05031279847025871, -0.04919631406664848, 0.0446428582072258, -0.0036399713717401028, -0.006186240818351507, -0.0012714045587927103, -0.06364525109529495]


In [8]:
# PART 1 — TASK 2

print("Embedding dimensions:", len(emb))

print("""
Observations:
- Fast inference
- Runs locally
- No API cost
- Good semantic similarity performance
""")


Embedding dimensions: 384

Observations:
- Fast inference
- Runs locally
- No API cost
- Good semantic similarity performance



# PART 1 — TASK 3

Comparison: OpenAI vs Hugging Face Embeddings
🔹 When to prefer OpenAI embeddings?

OpenAI embeddings are preferred when high semantic accuracy and strong generalization across domains are required. They are trained on very large and diverse datasets, which makes them effective for production-level applications such as enterprise search, customer support assistants, and large-scale RAG systems. OpenAI embeddings also require minimal setup since they are fully managed through APIs, reducing infrastructure complexity. They are ideal when reliability and performance are more important than cost.

🔹 When to prefer Hugging Face embeddings?

Hugging Face embeddings are preferred when working in offline or privacy-sensitive environments where data cannot be sent to external APIs. They are completely free and run locally, making them suitable for academic projects, experimentation, and small-scale applications. Developers also gain flexibility to choose or fine-tune models for specific domains. They are especially useful when budget constraints or customization requirements exist.

🔹 Cost vs Performance Trade-off

OpenAI embeddings generally provide better semantic understanding and retrieval accuracy but require paid API usage, which increases operational cost at scale. Hugging Face embeddings are free and locally deployable but may slightly underperform compared to proprietary models in complex reasoning or multilingual contexts. Therefore, the trade-off is between higher performance and convenience (OpenAI) versus lower cost and full control (Hugging Face).


In [9]:
# PART 2 — TASK 4

# Step 1: Convert chunks into embeddings & store in memory
vectorstore = FAISS.from_documents(chunks, hf_embeddings)

# Step 2: Define search function
def search(query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    return results

# Step 3: Test queries
queries = [
    "What is machine learning?",
    "Explain AI applications",
    "What is vector database?"
]

for q in queries:
    print("\nQuery:", q)
    docs = search(q)
    for d in docs:
        print("-", d.page_content[:120])



Query: What is machine learning?
- Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only 

Query: Explain AI applications
- Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only 

Query: What is vector database?
- Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only 


In [11]:
# PART 2 — TASK 5

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# NEW LangChain method
results = retriever.invoke("Explain embeddings")

for r in results:
    print(r.page_content[:150])


Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only on model memory, RAG retrieves


In [12]:
# PART 3 — TASK 6

ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

ollama_vector = ollama_embeddings.embed_query(
    chunks[0].page_content
)

print("Ollama embedding size:", len(ollama_vector))


C:\Users\kirut\AppData\Local\Temp\ipykernel_13276\3419092385.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(


Ollama embedding size: 768


# Compare HF vs Ollama

Comparison:
- Ollama runs fully local
- Better privacy
- Slightly slower first run
- No internet required



In [13]:
# PART 4 — TASK 7

faiss_db = FAISS.from_documents(chunks, hf_embeddings)

# Save locally
faiss_db.save_local("faiss_index")

print("FAISS index saved.")


FAISS index saved.


In [15]:
# Reload FAISS

loaded_faiss = FAISS.load_local(
    "faiss_index",
    hf_embeddings,
    allow_dangerous_deserialization=True
)

results = loaded_faiss.similarity_search("AI systems")

for r in results:
    print(r.page_content[:120])


Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only 


In [16]:
# PART 4 — TASK 8

chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    persist_directory="chroma_db"
)

chroma_db.persist()

print("ChromaDB stored.")


ChromaDB stored.


C:\Users\kirut\AppData\Local\Temp\ipykernel_13276\4251806789.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_db.persist()


In [17]:
results = chroma_db.similarity_search("What is embedding?")

for r in results:
    print(r.page_content[:120])


Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only 


# PART 4 — TASK 9

FAISS vs ChromaDB Comparison
🔹 In-Memory vs Persistent Storage

FAISS primarily operates as an in-memory vector search library optimized for extremely fast similarity computations. Although indexes can be saved and loaded, it is mainly designed for performance rather than database management. ChromaDB, on the other hand, is a persistent vector database that automatically stores embeddings on disk, allowing data to remain available across sessions without manual saving.

🔹 Use Cases for FAISS

FAISS is best suited for high-speed similarity search where performance is critical, such as research experiments, prototype RAG systems, or applications handling large embedding collections in memory. It is widely used when developers want maximum retrieval speed and full control over indexing behavior.

🔹 Use Cases for ChromaDB

ChromaDB is ideal for production-like applications where embeddings must persist between runs and include metadata filtering. It simplifies storage management and integrates easily with LangChain pipelines. Applications such as personal knowledge bases, document assistants, and long-term RAG systems benefit from ChromaDB’s persistence capabilities.



In [18]:
# PART 5 — TASK 10

def build_pipeline(embedding_type="hf", store_type="faiss"):

    if embedding_type == "hf":
        embedder = hf_embeddings
    else:
        embedder = ollama_embeddings

    if store_type == "faiss":
        db = FAISS.from_documents(chunks, embedder)
    else:
        db = Chroma.from_documents(
            chunks,
            embedder,
            persist_directory="chroma_db"
        )

    return db


pipeline = build_pipeline("hf", "faiss")

results = pipeline.similarity_search("Explain RAG systems")

for r in results:
    print(r.page_content[:150])


Retrieval Augmented Generation (RAG) combines information retrieval with large language models. Instead of relying only on model memory, RAG retrieves


# PART 5 — TASK 11

Observations & Insights
🔹 Importance of Embeddings in Generative AI

Embeddings transform text into numerical vector representations that capture semantic meaning rather than simple keywords. This allows systems to understand similarity between concepts even when wording differs. Embeddings form the foundation of modern retrieval systems by enabling machines to compare meaning mathematically.

🔹 Why Vector Databases Are Required

Vector databases efficiently store and search high-dimensional embedding vectors using similarity metrics like cosine similarity. Traditional databases cannot perform semantic similarity search effectively at scale. Vector databases enable fast retrieval of the most relevant documents, which is essential for real-time AI applications.

🔹 How This Pipeline Enables RAG Systems

The pipeline converts documents into embeddings, stores them in a vector database, and retrieves relevant context during a user query. This retrieved information is then supplied to a language model to generate accurate and context-aware responses. By combining retrieval with generation, RAG systems reduce hallucinations and allow models to answer questions using external knowledge sources.
